## data_utils: convert_csv_to_h5.py

In [ ]:
import pandas as pd
import h5py

DATA_DIR = 'data/'

def convert_csv_to_h5(filename):
    print 'converting', filename
    data_frame = pd.read_csv(DATA_DIR + filename, header=None)
    h5_file = h5py.File(DATA_DIR + filename[:-3] + 'h5', 'w')
    h5_file.create_dataset('data', data = data_frame.as_matrix())
    h5_file.close()

for fn in ['fALFF.csv', 'coord.csv', 'region_code.csv']:
    convert_csv_to_h5(DATA_DIR + fn)


## data_utils: prepare_time_series.py

In [ ]:
import pandas as pd
import h5py
import numpy as np
import os
from tqdm import tqdm

DATA_DIR = 'data/time_series/'
NUM_BRAIN_REGIONS = 116
filenames = os.listdir(DATA_DIR)
corr_file = h5py.File('data/correlation.h5', 'w')
lengths = {}
eeg = {}
max_length = 0
for i, f in enumerate(tqdm(filenames)):
    id = f[:5]
    data = pd.read_csv(DATA_DIR + f, header=None).as_matrix()
    data = np.reshape(data, (NUM_BRAIN_REGIONS, len(data)/NUM_BRAIN_REGIONS))
    correlation = np.corrcoef(data)
    correlation = np.nan_to_num(correlation)
    corr_file.create_dataset(id, data = correlation)
    eeg[id] = data
    if data.shape[1] > max_length:
        max_length = data.shape[1] 
    
print max_length
eeg_out = []
lengths_out = []
labels_out = []
phenotype_data = pd.read_csv('data/phenotype_data.csv', header=None)
phenotype_data = phenotype_data.as_matrix()
labels = np.array(phenotype_data[:,2] - 1, dtype=int)
ids = np.array(phenotype_data[:,1], dtype=str)

for i, id in enumerate(ids):
    if id not in eeg: continue
    eg = eeg[id]
    length = eg.shape[1]
    eg = np.pad(eg, ((0, 0), (0, max_length - length)), 'constant', constant_values=(0))
    print eg.shape
    label = labels[i]
    eeg_out.append(eg)
    labels_out.append(label)
    lengths_out.append(length)
train_split = 0.7
val_split = 0.1
test_split = 0.2
round_num_examples = len(eeg_out) - len(eeg_out) % 10
train_num = int(round_num_examples * train_split)
val_num = int(round_num_examples * val_split)
test_num = int(round_num_examples * test_split + len(eeg_out) % 10)
print train_num, val_num, test_num, len(eeg_out)

def save_to_h5(files, name, data):
    train = data[:train_num]
    val = data[train_num:train_num+val_num]
    test = data[:test_num]
    files[0].create_dataset(name, data = train)
    files[1].create_dataset(name, data = val)
    files[2].create_dataset(name, data = test)
train_file = h5py.File('data/eeg/train.h5', 'w')
val_file = h5py.File('data/eeg/val.h5', 'w')
test_file = h5py.File('data/eeg/test.h5', 'w')

save_to_h5([train_file, val_file, test_file], 'eeg', np.stack(eeg_out))
save_to_h5([train_file, val_file, test_file], 'labels', np.stack(labels_out))
save_to_h5([train_file, val_file, test_file], 'lengths', np.stack(lengths_out))
train_file.close()
val_file.close()
test_file.close()
corr_file.close()


## nn_utils.py

In [ ]:
import tensorflow as tf
import numpy as np
slim = tf.contrib.slim

def conv3d(inputs, kernel_size, num_channels, num_filters,
        scope='', stride=1, activation=tf.nn.relu, l2=0.0, padding='SAME', trainable=True):
    with tf.variable_scope(scope, initializer = slim.xavier_initializer()):
        weights = tf.get_variable('weights',
                ([kernel_size, kernel_size, kernel_size, num_channels, num_filters]), trainable=trainable)
        output = tf.nn.conv3d(inputs, weights, [1, stride, stride, stride, 1], padding)
    output = activation(output)
    output = slim.batch_norm(output)
    reg = l2*tf.nn.l2_loss(weights)
    tf.add_to_collection(tf.GraphKeys.REGULARIZATION_LOSSES, reg)
    return output

def conv3d_transpose(inputs, kernel_size, num_channels, num_filters,
        scope='', stride=1, activation=tf.nn.relu, l2=0.0, padding='SAME'):
    def get_deconv_dim(dim_size, stride_size, kernel_size, padding):
        dim_size *= stride_size
        if padding == 'VALID' and dim_size is not None:
            dim_size += max(kernel_size - stride_size, 0)
        return dim_size
    with tf.variable_scope(scope, initializer = slim.xavier_initializer(), reuse=True):
        weights = tf.get_variable('weights',
                ([kernel_size, kernel_size, kernel_size, num_filters, num_channels]))
        batch_size, height, width, depth, _ = inputs.get_shape()
        out_height = get_deconv_dim(height, stride, kernel_size, padding)
        out_width = get_deconv_dim(width, stride, kernel_size, padding)
        out_depth = get_deconv_dim(depth, stride, kernel_size, padding)
        output_shape = tf.pack([tf.shape(inputs)[0], int(out_height), int(out_width), int(out_depth), int(num_filters)])
        output = tf.nn.conv3d_transpose(inputs, weights, output_shape, [1, stride, stride, stride, 1], padding=padding)
        out_shape = inputs.get_shape().as_list()
        out_shape[-1] = num_filters
        out_shape[1] = get_deconv_dim(out_shape[1], stride, kernel_size, padding)
        out_shape[2] = get_deconv_dim(out_shape[2], stride, kernel_size, padding)
        out_shape[3] = get_deconv_dim(out_shape[3], stride, kernel_size, padding)
        output.set_shape(out_shape)
    output = activation(output)
    output = slim.batch_norm(output)
    reg = l2*tf.nn.l2_loss(weights)
    tf.add_to_collection(tf.GraphKeys.REGULARIZATION_LOSSES, reg)
    return output
def rotate_image_tensor(image, angle, mode='black'):
    s = image.get_shape().as_list()
    assert len(s) == 3, "Input needs to be 3D."
    assert (mode=='repeat')or(mode=='black')or(mode=='white')or(mode=='ones'),"Unknown boundary mode."
    image_center = [np.floor(x/2) for x in s]
    coord1 = tf.range(s[0])
    coord2 = tf.range(s[1])
    coord1_vec = tf.tile(coord1, [s[1]])
    coord2_vec_unordered = tf.tile(coord2, [s[0]])
    coord2_vec_unordered = tf.reshape(coord2_vec_unordered, [s[0], s[1]])
    coord2_vec = tf.reshape(tf.transpose(coord2_vec_unordered, [1, 0]), [-1])
    coord1_vec_centered = coord1_vec - image_center[0]
    coord2_vec_centered = coord2_vec - image_center[1]
    coord_new_centered = tf.cast(tf.pack([coord1_vec_centered, coord2_vec_centered]), tf.float32)
    rot_mat_inv = tf.dynamic_stitch([0, 1, 2, 3], [tf.cos(angle), tf.sin(angle), -tf.sin(angle), tf.cos(angle)])
    rot_mat_inv = tf.reshape(rot_mat_inv, shape=[2, 2])
    coord_old_centered = tf.matmul(rot_mat_inv, coord_new_centered)
    coord1_old_nn = tf.cast(tf.round(coord_old_centered[0, :] + image_center[0]), tf.int32)
    coord2_old_nn = tf.cast(tf.round(coord_old_centered[1, :] + image_center[1]), tf.int32)
    if mode == 'repeat':
        coord_old1_clipped = tf.minimum(tf.maximum(coord1_old_nn, 0), s[0]-1)
        coord_old2_clipped = tf.minimum(tf.maximum(coord2_old_nn, 0), s[1]-1)
    else:
        outside_ind1 = tf.logical_or(tf.greater(coord1_old_nn, s[0]-1), tf.less(coord1_old_nn, 0))
        outside_ind2 = tf.logical_or(tf.greater(coord2_old_nn, s[1]-1), tf.less(coord2_old_nn, 0))
        outside_ind = tf.logical_or(outside_ind1, outside_ind2)
        coord_old1_clipped = tf.boolean_mask(coord1_old_nn, tf.logical_not(outside_ind))
        coord_old2_clipped = tf.boolean_mask(coord2_old_nn, tf.logical_not(outside_ind))
        coord1_vec = tf.boolean_mask(coord1_vec, tf.logical_not(outside_ind))
        coord2_vec = tf.boolean_mask(coord2_vec, tf.logical_not(outside_ind))
    coord_old_clipped = tf.cast(tf.transpose(tf.pack([coord_old1_clipped, coord_old2_clipped]), [1, 0]), tf.int32)
    coord_new = tf.transpose(tf.cast(tf.pack([coord1_vec, coord2_vec]), tf.int32), [1, 0])
    image_channel_list = tf.split(2, s[2], image)
    image_rotated_channel_list = list()
    for image_channel in image_channel_list:
        image_chan_new_values = tf.gather_nd(tf.squeeze(image_channel), coord_old_clipped)
        if (mode == 'black') or (mode == 'repeat'):
            background_color = 0
        elif mode == 'ones':
            background_color = 1
        elif mode == 'white':
            background_color = 255
        image_rotated_channel_list.append(tf.sparse_to_dense(coord_new, [s[0], s[1]], image_chan_new_values,
                                                             background_color, validate_indices=False))
    image_rotated = tf.transpose(tf.pack(image_rotated_channel_list), [1, 2, 0])
    return image_rotated
def get_save_path(config):
    path = 'weights/model'
    for a, v in config.__dict__.iteritems():
        path += '_' + a + '=' + str(v)
    return path


## mri_input.py

In [ ]:
import tensorflow as tf
import numpy as np
from nn_utils import rotate_image_tensor
IMAGE_DIMS = [96, 112, 96]
NOISE_STD = 0.1
def distort_image(image, rotate, noise):
    axis = 0
    if rotate:
        image = tf.expand_dims(image, -1)
        image = tf.split(axis, image.get_shape()[axis].value, image)
        image = [tf.squeeze(im, squeeze_dims=[axis]) for im in image]
        angle = tf.random_uniform((), 0, 2*3.141)
        image = [rotate_image_tensor(im, angle) for im in image]
        image = tf.pack(image)
        image = tf.squeeze(image)
    noise = tf.random_normal(image.get_shape(), stddev=noise)
    image += noise
    return image
def process_image(image, downsample_factor, train, rotate, noise):
    image = tf.reshape(image, IMAGE_DIMS)
    if downsample_factor > 1:
        image = tf.expand_dims(image, -1)
        image = tf.expand_dims(image, 0)
        image = tf.nn.avg_pool3d(image, 
                [1,downsample_factor,downsample_factor,downsample_factor,1],
                [1,downsample_factor,downsample_factor,downsample_factor,1],
                'SAME')
        image = tf.squeeze(image)
    if train:
        image = distort_image(image, rotate, noise)
    return image
def read_and_decode_single_example(filename_queue, train=True,
        downsample_factor=1, corr=0, rotate=True, noise=0):
    reader = tf.TFRecordReader()
    _, serialized_example = reader.read(filename_queue)
    features = tf.parse_single_example(
        serialized_example,
        features={
            'label': tf.FixedLenFeature([], tf.int64),
            'sex': tf.FixedLenFeature([], tf.int64),
            'image': tf.FixedLenFeature(np.prod(IMAGE_DIMS), tf.float32),
            'corr': tf.FixedLenFeature(116**2, tf.float32)
        })
    label = features['label']
    sex = features['sex']
    corr = features['corr']
    corr = tf.reshape(corr, [116, 116])
    image = features['image']
    if corr != 2:
        image = process_image(image, downsample_factor, train, rotate, noise)
    return image, label, sex, corr


## train_cnn.py

In [ ]:
import tensorflow as tf
import numpy as np
import argparse
import time
import os
from cnn_3d import CNN_3D, Config
import mri_input
from test_cnn import test_cnn
import nn_utils
import sys
BATCH_SIZE = 15
MAX_STEPS = 10000
SAVE_EVERY = 10
MIN_EXAMPLES_IN_QUEUE = 1000
EARLY_STOPPING = 50
def train_cnn(config):
    mode = config.mode
    save_path = nn_utils.get_save_path(config)
    intro_str = '==> Building 3D CNN with %d layers'
    print intro_str % (config.num_layers)
    if config.use_sex_labels:
        print 'Debugging by training on gender labels'
    fn = 'data/mri_{}train.tfrecords'.format(config.gate)
    print '==> Reading examples from', fn
    filename_queue = tf.train.string_input_producer([fn], num_epochs=None)
    with tf.device('/cpu:0'):
        image, label, sex, corr = mri_input.read_and_decode_single_example(filename_queue,
                downsample_factor=config.downsample_factor, corr=config.use_correlation,
                rotate=config.rotate, noise=config.noise)
        image_batch, label_batch, sex_batch, corr_batch = tf.train.shuffle_batch(
            [image, label, sex, corr], batch_size=BATCH_SIZE,
            capacity=10000,min_after_dequeue=MIN_EXAMPLES_IN_QUEUE)
    label_batch = sex_batch if config.use_sex_labels else label_batch 
    cnn = CNN_3D(config,image_batch,label_batch,corr_batch)
    pretrained_names = ['conv_' + str(i+1) + '/weights:0' for i in range(config.num_layers_to_restore)]
    pretrained_vars = [v for v in tf.all_variables() if v.name in pretrained_names]
    print '==> variables to be restored:'
    for v in pretrained_vars:
        print v.name
    print '==> variables to be trained:'
    for v in tf.trainable_variables():
        print v.name
    sess = tf.Session()
    summary_writer = tf.train.SummaryWriter('summaries/' + config.sum_dir + 
                                        '/{}/train'.format(save_path[8:]), sess.graph)
    saver = tf.train.Saver()
    coord = tf.train.Coordinator()
    threads = tf.train.start_queue_runners(sess=sess, coord=coord)
    train_accuracy = 0
    best_val_loss = None
    for step in xrange(MAX_STEPS):
        start_time = time.time()
        _, loss_value, image_value, output_value, accuracy, summary = 
        sess.run([cnn.train_op,cnn.loss,image_batch,cnn.outputs,cnn.merged])
        train_accuracy += accuracy
        duration = time.time() - start_time
        summary_writer.add_summary(summary, step)
        if step % 2 == 0:
            num_examples_per_step = BATCH_SIZE
            sec_per_batch = float(duration)
            format_str = ('step %s, loss = %.2f (%.3f''sec/batch)')
            s = format_str % (step, loss_value, sec_per_batch)
            sys.stdout.write('\r' + s + ' ')
            sys.stdout.flush()
        if (step % SAVE_EVERY == 0 or (step + 1) == MAX_STEPS) and step != 0:
            if mode == 'supervised':
                saver.save(sess, save_path)
                print '==> evaluating valid and train accuracy'
                val_accuracy, val_loss = test_cnn(config, start_step=step)
                print 'train accuracy:', train_accuracy/float(SAVE_EVERY) 
                print 'val accuracy:', val_accuracy
                train_accuracy = 0
                if val_loss < best_val_loss or best_val_loss is None:
                    early_stopping_count = 0
                    best_val_loss = val_loss
                    print '==> saving weights to', save_path + '_best'
                    saver.save(sess, save_path + '_best')
                else:
                    early_stopping_count += 1
                    if early_stopping_count >= EARLY_STOPPING: break
if __name__ == '__main__':
    parser = argparse.ArgumentParser()
    parser.add_argument("-m", "--mode", default="supervised")
    parser.add_argument("-l", "--num-layers", type=int, default=4)
    parser.add_argument("-r", "--num-layers-to-restore", type=int, default=0)
    parser.add_argument("-t", "--num-layers-to-train", type=int, default=4, 
            help="trains the specified number of innermost layers")
    parser.add_argument("-d", "--downsample_factor", type=int, default=2)
    parser.add_argument("-s", "--use_sex_labels", type=bool, default=False)
    parser.add_argument("-c", "--use_correlation", type=int, default=0, help="0 
                indicates no use, 1 supplements, 2 trains on only correlation")
    parser.add_argument('-g', '--gate', default='')
    parser.add_argument('-sd', '--sum-dir', default='')
    parser.add_argument('-ro', '--rotate', type=bool, default=True)
    parser.add_argument('-no', '--noise', type=float, default=0.1)
    args = parser.parse_args()
    config = Config()
    config.gate = args.gate
    config.num_layers = args.num_layers
    config.num_layers_to_train = args.num_layers_to_train
    config.mode = args.mode
    config.num_layers_to_restore = args.num_layers_to_restore
    config.downsample_factor = args.downsample_factor
    config.use_sex_labels = args.use_sex_labels
    config.use_correlation = args.use_correlation
    config.sum_dir = args.sum_dir
    config.rotate = args.rotate
    config.noise = args.noise
    train_cnn(config)

       


## cnn_3d.py

In [ ]:
import tensorflow as tf
import numpy as np
import h5py
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from nn_utils import conv3d, conv3d_transpose
import sys
import time
slim = tf.contrib.slim
DATA_DIR = 'data'
class Config():
    lr = 0.001
    l2 = 0.01
    dropout = 0.8
    kernel_size = 3 
    num_filters = 8
    downsample_every = 2
    num_layers = 4
    num_layers_to_train = num_layers
    num_layers_to_restore = 0
    downsample_every = 2
    downsample_factor = 2
    use_sex_labels = False
    mode = 'pretrain'
class CNN_3D(object):
    def _save_images(self, images, outputs, name):
        if len(outputs.shape) == 4:
            outputs = np.expand_dims(outputs, 4)
        im_depth = images.shape[1]/2
        out_depth = outputs.shape[1]/2
        fig = plt.figure()
        a = fig.add_subplot(1,2,1)
        imgplot = plt.imshow(images[0,im_depth,:,:])
        a.set_title('Original')
        a = fig.add_subplot(1,2,2)
        imgplot = plt.imshow(outputs[0,out_depth,:,:,0])
        a.set_title('Reconstructed')
        fig.savefig('figures/' + name + ".png")
        plt.close()
    def make_predictions(self, output):
        """Get answer predictions from output"""
        preds = tf.nn.softmax(output)
        pred = tf.argmax(preds, 1)
        return pred
    
    def calc_accuracy(self, predictions, labels):
        correct_prediction = tf.equal(predictions, labels)
        accuracy = tf.reduce_mean(tf.cast(correct_prediction, tf.float32))
        return accuracy
    def calc_loss(self, outputs, labels):
        if self.config.mode == 'pretrain':
            loss = tf.reduce_sum(tf.square(outputs - labels))
        else:
            loss = tf.reduce_sum(tf.nn.sparse_softmax_cross_entropy_with_logits(outputs, labels)) 
        loss += tf.reduce_sum(tf.pack(tf.get_collection(tf.GraphKeys.REGULARIZATION_LOSSES)))
        return loss
    def add_train_op(self, loss):
        train_op = tf.train.AdamOptimizer(learning_rate=self.config.lr).minimize(loss)
        return train_op
    def correlation_inference(self, correlation):
        flattened = slim.flatten(correlation)
        layer_1 = slim.fully_connected(flattened, 1000, weights_regularizer=slim.l2_regularizer(self.config.l2))
        output = slim.fully_connected(layer_1, 2, activation_fn=None, weights_regularizer=slim.l2_regularizer(self.config.l2))
        return output
    def image_inference(self, images, train=True):
        images = tf.expand_dims(images, -1)
        num_filters = self.config.num_filters
        kernel_size = self.config.kernel_size
        downsample_every = self.config.downsample_every
        num_layers = self.config.num_layers
        num_layers_to_train = self.config.num_layers_to_train
        if num_layers > 0:
            trainable = True if num_layers_to_train >= num_layers else False
            forward = conv3d(images, kernel_size, 1, num_filters,
                    scope='conv_1', trainable=trainable)
        else:
            forward = images
        for i in range(num_layers - 1):
            stride = 2 if i % downsample_every == 0 else 1
            trainable = True if i+2 > num_layers - num_layers_to_train else False
            forward = conv3d(forward, kernel_size, num_filters, num_filters,
                    scope='conv_' + str(i+2), stride=stride, trainable=trainable)
            if stride == 2:
                forward = slim.dropout(forward, keep_prob=self.config.dropout, is_training=train)
        if self.config.mode == 'pretrain':
            backward = forward
            for i in range(num_layers - 1):
                stride = 1 if i % downsample_every == 0 else 2
                backward = conv3d_transpose(backward, kernel_size, num_filters, num_filters,
                        scope='conv_' + str(num_layers - i), stride=stride)
            backward = conv3d_transpose(backward, kernel_size, num_filters, 1, scope='conv_1', stride=2)
            output = tf.squeeze(backward)
        else:
            flattened = slim.flatten(forward)        
            with tf.variable_scope('fully_connected'):
                output = slim.fully_connected(flattened, 2000, weights_regularizer=slim.l2_regularizer(self.config.l2))
                output = slim.fully_connected(output, 500, weights_regularizer=slim.l2_regularizer(self.config.l2))
                output = slim.fully_connected(output, 2, activation_fn=None, weights_regularizer=slim.l2_regularizer(self.config.l2))
        return output, forward
    def inference(self, images, correlation, train):
        if self.config.use_correlation == 2: 
            corr_outputs = self.correlation_inference(correlation)
            return corr_outputs
        image_outputs, self.filt = self.image_inference(images, train)
        if self.config.use_correlation == 1:
            corr_outputs = self.correlation_inference(correlation)
            outputs = tf.concat(1, [image_outputs, corr_outputs])
            outputs = slim.fully_connected(outputs, 2, activation_fn=None)
            return outputs
        return image_outputs
    def add_summaries(self, images, train):
        dataset = 'train' if train else 'validation'
        tf.scalar_summary('loss', self.loss)
        tf.scalar_summary('accuracy', self.accuracy)
        if self.config.use_correlation != 2:
            filt_depth = int(self.filt.get_shape()[1])/2
            im_depth = int(images.get_shape()[1])/2
            filt = tf.squeeze(
          tf.slice(self.filt, [0, filt_depth, 0, 0, 0], [-1, 1, -1, -1, 1]),[1])
            image = tf.squeeze(tf.slice(images,[0,im_depth,0,0],[-1,1,-1,-1]))
            tf.image_summary('filter', filt)
            tf.image_summary('image', tf.expand_dims(image, -1))
    def __init__(self, config, image_batch, label_batch, corr_batch, train=True):
        self.config = config
        label_batch = image_batch if config.mode == 'pretrain' else label_batch
        self.outputs = self.inference(image_batch, corr_batch, train)
        self.loss = self.calc_loss(self.outputs, label_batch)
        self.predictions = self.make_predictions(self.outputs)
        self.accuracy = self.calc_accuracy(self.predictions, label_batch)
        self.train_op = self.add_train_op(self.loss)
        self.add_summaries(image_batch, train)
        self.merged = tf.merge_all_summaries()
        



## test_cnn.py

In [ ]:
import tensorflow as tf
import numpy as np
import os
import argparse
from cnn_3d import CNN_3D, Config
import mri_input
import nn_utils
BATCH_SIZE = 15
NUM_EXAMPLES = {'train': 749, 'val': 107, 'test': 215}
def test_cnn(config, dataset='val', start_step=0, restore_path=None):
    test_graph = tf.Graph()
    best = True if dataset == 'test' else False
    if restore_path is None:
        restore_path = nn_utils.get_save_path(config)
    if best:
        restore_path += '_best'
    print 'restore path:', restore_path
    with test_graph.as_default():
        fn = 'data/mri_{}.tfrecords'.format(config.gate + dataset)
        filename_queue = tf.train.string_input_producer([fn], num_epochs=1)
        with tf.device('/cpu:0'):
            image, label, sex, corr = mri_input.read_and_decode_single_example(filename_queue, train=False,
                    downsample_factor=config.downsample_factor)
            image_batch, label_batch, sex_batch, corr_batch = tf.train.batch(
                [image, label, sex, corr], batch_size=BATCH_SIZE,
                capacity=100,allow_smaller_final_batch=True)
        label_batch = sex_batch if config.use_sex_labels else label_batch 
        cnn = CNN_3D(config,image_batch,label_batch,corr_batch,train=False)
        sess = tf.Session()
        summary_writer = tf.train.SummaryWriter('summaries/' + config.sum_dir + '/{}/test'.format(restore_path[8:]))
        init = tf.group(tf.initialize_all_variables(), tf.initialize_local_variables())
        sess.run(init)
        restorer = tf.train.Saver()
        assert os.path.exists(restore_path)
        print 'restoring from', restore_path
        restorer.restore(sess, restore_path)
        tf.train.start_queue_runners(sess=sess)
        step = 0
        val_accuracy = 0
        overall_loss = 0
        try:
            while True:
                loss_value, accuracy, output_val, summary = sess.run([
                    cnn.loss,cnn.accuracy,cnn.outputs,cnn.merged])
                val_accuracy += accuracy
                overall_loss += loss_value
                summary_writer.add_summary(summary, start_step + step)
                step += 1
        except tf.errors.OutOfRangeError:
            return val_accuracy/float(step), overall_loss/float(step)
if __name__ == '__main__':
    parser = argparse.ArgumentParser()
    parser.add_argument('-ds', '--dataset-split', default='val')
    parser.add_argument("-m", "--mode", default="supervised")
    parser.add_argument("-l", "--num-layers", type=int, default=4)
    parser.add_argument("-r", "--num-layers-to-restore", type=int, default=0)
    parser.add_argument("-t", "--num-layers-to-train", type=int, default=4, 
            help="trains the specified number of innermost layers")
    parser.add_argument("-d", "--downsample_factor", type=int, default=2)
    parser.add_argument("-s", "--use_sex_labels", type=bool, default=False)
    parser.add_argument("-c", "--use_correlation", type=int, default=0, 
                        help="0 indicates no use, 1 supplements, 2 trains on only correlation")
    parser.add_argument('-g', '--gate', default='')
    parser.add_argument('-ro', '--rotate', type=int, default=1)
    parser.add_argument('-no', '--noise', type=float, default=0.1)
    parser.add_argument('-sd', '--sum-dir', default='')
    parser.add_argument('--restore_path', default=None)
    args = parser.parse_args()
    config = Config()
    config.gate = args.gate
    config.num_layers = args.num_layers
    config.num_layers_to_train = args.num_layers_to_train
    config.mode = args.mode
    config.num_layers_to_restore = args.num_layers_to_restore
    config.downsample_factor = args.downsample_factor
    config.use_sex_labels = args.use_sex_labels
    config.use_correlation = args.use_correlation
    config.sum_dir = args.sum_dir
    config.rotate = bool(args.rotate)
    if args.noise == 1: args.noise = int(args.noise)
    config.noise = args.noise
    accuracy, loss = test_cnn(config, dataset=args.dataset_split, restore_path=args.restore_path)
    print args.dataset_split, 'accuracy:', accuracy
    print args.dataset_split, 'loss:', loss





## experiments.py

In [ ]:
from train_cnn import train_cnn
from cnn_3d import Config
def compare_gating():
    config = Config()
    config.downsample_factor = 6
    config.num_layers = 0
    config.num_layers_to_train = 0
    config.mode = 'supervised'
    config.num_layers_to_restore = 0
    config.use_correlation = 0
    config.sum_dir = 'gating_comparison'
    config.use_sex_labels = False
    config.gate = 'male'
    train_cnn(config)
    config.use_sex_labels = True
    config.gate = 'equal_gender'
    train_cnn(config)
    config.use_sex_labels = False
    config.gate = 'shuffle'
    train_cnn(config)
def compare_layers():
    config = Config()
    config.downsample_factor = 6
    config.num_layers = 0
    config.num_layers_to_train = 0
    config.mode = 'supervised'
    config.num_layers_to_restore = 0
    config.use_correlation = 0
    config.sum_dir = 'layer_comparison'
    config.use_sex_labels = False
    config.gate = 'male'
    train_cnn(config)
    config.downsample_factor = 4
    config.num_layers = 2
    config.num_layers_to_train = 2
    train_cnn(config)
    config.downsample_factor = 2
    config.num_layers = 4
    config.num_layers_to_train = 4
    train_cnn(config)
    config.downsample_factor = 0
    config.num_layers = 6
    config.num_layers_to_train = 6
    train_cnn(config)
def compare_data_augmentation_and_pretraining():
    config = Config()
    config.downsample_factor = 0
    config.num_layers = 6
    config.num_layers_to_train = 6
    config.mode = 'supervised'
    config.num_layers_to_restore = 0
    config.use_correlation = 0
    config.sum_dir = 'data_augmentation_comparison'
    config.use_sex_labels = False
    config.gate = 'male'
    config.num_layers_to_restore = 6
    config.num_layers_to_train = 0
    config.rotate = True
    config.noise = 0.1
    train_cnn(config)
    config.num_layers_to_train = 6
    config.rotate = True
    config.noise = 0.1
    train_cnn(config)
    config.num_layers_to_restore = 0
    config.num_layers_to_train = 6
    config.rotate = True
    config.noise = 0.1
    train_cnn(config)
    config.rotate = False
    config.noise = 0
    train_cnn(config)
    config.rotate = False
    config.noise = 1
    train_cnn(config)
def compare_correlation():
    config = Config()
    config.use_correlation = 1
    config.gate = 'male'
    config.sum_dir = 'correlation_comparison'
    config.rotate = True
    config.noise = 0.1
    config.num_layers = 6
    config.num_layers_to_restore = 6
    config.num_layers_to_train = 0
    config.mode = 'supervised'
    train_cnn(config)
compare_correlation()
